In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "catcher-feedback-eval"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 1. 단건 테스트 — generate_weekly_feedback()

week_start ~ week_end 기간의 소비 데이터를 분석해 주간 피드백을 생성합니다.

In [ ]:
from catcher_llm.services.consumption_feedback.weekly_feedback import generate_weekly_feedback

result = generate_weekly_feedback(
    member_id=1,
    week_start="2024-01-08",
    week_end="2024-01-14"
)

if result.error:
    print("오류:", result.error)
else:
    print("[주간 피드백]")
    print(result.feedback.feedback_message)

# 2. target 함수 정의

In [ ]:
def target(inputs: dict):
    result = generate_weekly_feedback(
        member_id=inputs["member_id"],
        week_start=inputs["week_start"],
        week_end=inputs["week_end"]
    )

    if result.error:
        return {"answer": f"[ERROR] {result.error}", "week_total": 0, "top_category": ""}

    week_total = (
        result.weekly_analysis.stable_metrics.week_total
        if result.weekly_analysis else 0
    )
    top_category = (
        result.weekly_analysis.stable_metrics.worst_category
        if result.weekly_analysis else ""
    )

    return {
        "answer": result.feedback.feedback_message,
        "week_total": week_total,
        "top_category": str(top_category)
    }

# 3. LangSmith Dataset 생성

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "catcher-feedback-weekly-eval"

examples = [
    {
        "inputs": {"member_id": 1, "week_start": "2024-01-08", "week_end": "2024-01-14"},
        "outputs": {"expected_trait": "이번 주 총지출과 전주 대비 변화를 언급하고, 가장 많이 쓴 카테고리의 패턴을 짚어야 한다."}
    },
    {
        "inputs": {"member_id": 2, "week_start": "2024-01-15", "week_end": "2024-01-21"},
        "outputs": {"expected_trait": "반복 소비 패턴(요일·시간대)을 파악하고 다음 주 규칙 하나를 제안해야 한다."}
    },
    {
        "inputs": {"member_id": 3, "week_start": "2024-02-05", "week_end": "2024-02-11"},
        "outputs": {"expected_trait": "절약 중인 유저이므로 잘한 점을 인정하면서 추가 개선 포인트를 제안해야 한다."}
    },
    {
        "inputs": {"member_id": 4, "week_start": "2024-02-19", "week_end": "2024-02-25"},
        "outputs": {"expected_trait": "구독 서비스 지출 패턴을 파악하고 중복 구독 점검을 제안해야 한다."}
    },
    {
        "inputs": {"member_id": 1, "week_start": "2024-03-04", "week_end": "2024-03-10"},
        "outputs": {"expected_trait": "주 중반 이후 소비가 급증하는 패턴이 있다면 원인을 짚고 다음 주 대응 전략을 제안해야 한다."}
    },
]

existing = [d for d in client.list_datasets() if d.name == dataset_name]
if existing:
    dataset = existing[0]
    print(f"기존 dataset 사용: {dataset.name}")
else:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Catcher LLM 주간 피드백 품질 평가"
    )
    for ex in examples:
        client.create_example(
            inputs=ex["inputs"],
            outputs=ex["outputs"],
            dataset_id=dataset.id
        )
    print(f"새 dataset 생성: {dataset.name} ({len(examples)}개)")

# 4. Evaluator 정의

| Evaluator | 방식 | 주간 특화 기준 |
|---|---|---|
| `groundedness` | LLM judge | 주간 총지출·카테고리가 피드백에 반영됐는가 |
| `pattern_detection` | LLM judge | 반복 패턴(요일/시간/카테고리)을 발견했는가 |
| `actionability` | LLM judge | 다음 주 실천 가능한 규칙을 제안했는가 |
| `contains_amount` | Heuristic | 구체적 금액이 포함되어 있는가 |

In [ ]:
import re
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def groundedness_evaluator(run, example):
    answer = run.outputs.get("answer", "")
    week_total = run.outputs.get("week_total", 0)
    top_category = run.outputs.get("top_category", "")

    prompt = f"""
아래 주간 피드백이 실제 소비 데이터에 근거하는지 0~1로 평가해줘.
이번 주 총지출({week_total:,}원), 최다 지출 카테고리({top_category})를 반영하면 높은 점수야.
숫자 하나만 출력해.

피드백: {answer}
"""
    score = judge_llm.invoke(prompt).content.strip()
    try:
        return {"key": "groundedness", "score": float(score)}
    except ValueError:
        return {"key": "groundedness", "score": 0.0}


def pattern_detection_evaluator(run, example):
    """반복 소비 패턴(요일·시간대·카테고리)을 발견하고 설명하는가"""
    answer = run.outputs.get("answer", "")

    prompt = f"""
아래 주간 피드백이 이번 주 반복 소비 패턴(특정 요일, 시간대, 카테고리 반복 등)을
구체적으로 발견하고 설명하는지 0~1로 평가해줘.
단순한 총액 나열은 낮게, 패턴을 언급하면 높게.
숫자 하나만 출력해.

피드백: {answer}
"""
    score = judge_llm.invoke(prompt).content.strip()
    try:
        return {"key": "pattern_detection", "score": float(score)}
    except ValueError:
        return {"key": "pattern_detection", "score": 0.0}


def actionability_evaluator(run, example):
    answer = run.outputs.get("answer", "")
    expected = example.outputs.get("expected_trait", "")

    prompt = f"""
아래 주간 피드백이 다음 주 바로 실천할 수 있는 구체적 행동(규칙/제한/대안)을
제안하는지 0~1로 평가해줘.

기대 특성: {expected}
피드백: {answer}

숫자 하나만 출력해.
"""
    score = judge_llm.invoke(prompt).content.strip()
    try:
        return {"key": "actionability", "score": float(score)}
    except ValueError:
        return {"key": "actionability", "score": 0.0}


def contains_amount_evaluator(run, example):
    answer = run.outputs.get("answer", "")
    has_amount = bool(re.search(r'\d[\d,]*\s*(원|만원|만\s*원)', answer))
    return {"key": "contains_amount", "score": 1 if has_amount else 0}


print("evaluator 4개 정의 완료")

# 5. evaluate() 실행 → LangSmith 반영

In [ ]:
from langsmith.evaluation import evaluate

evaluate(
    target,
    data=dataset_name,
    evaluators=[
        groundedness_evaluator,
        pattern_detection_evaluator,
        actionability_evaluator,
        contains_amount_evaluator,
    ],
    experiment_prefix="weekly-feedback-v1"
)